In [1]:
!pip install mediapipe av
!pip install dpkt
# !pip install sdtlib 
# !wget -q -O blaze_face_short_range.tflite -q https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/1/blaze_face_short_range.tflite
!wget -O face_landmarker.task -q https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task
!wget -O selfie_segmenter.tflite -q https://storage.googleapis.com/mediapipe-models/image_segmenter/selfie_segmenter/float16/latest/selfie_segmenter.tflite

from IPython.display import clear_output
clear_output(wait=False)

# %%writefile main.py

In [2]:
%%writefile main.py
#!/usr/bin/env python3
"""
decode_rtp_h264_v7.py
=====================
Decode RTP H264 stream từ PCAP/PCAPNG IMS video call → MP4.

Lưu ý quan trọng:
    - Một số bản capture bị export text-mode khiến byte 0x00 trong PCAP
      header bị thành 0x20. Khi đó dpkt không nhận ra RTP và video bị mờ/xám.
    - Chỉ sửa 0x20 → 0x00 ở PCAP global header (24 bytes) và mỗi packet
      header (16 bytes). KHÔNG động vào payload để tránh hỏng H264.

Usage:
    pip install dpkt
    python3 decode_rtp_h264_v7.py <input.pcap|input.pcapng> [output.mp4]
            [--ssrc 0xXXXX] [--stream recv|send] [--repair-space-nulls]
            (--repair-space-nulls ép sửa header kể cả khi auto không phát hiện)

Requires: dpkt, ffmpeg
"""

import dpkt, io, struct, os, sys, subprocess, re, shutil
from collections import defaultdict

RTP_CLOCK = 90000


# ─────────────────────────────────────────────
#  STEP 1: Parse pcap
# ─────────────────────────────────────────────

def _fix_space_bytes(buf: bytearray, start: int, end: int) -> bool:
    """Convert 0x20 → 0x00 trong vùng [start, end)."""
    changed = False
    for i in range(start, min(end, len(buf))):
        if buf[i] == 0x20:
            buf[i] = 0x00
            changed = True
    return changed


def repair_pcap_headers(raw: bytes) -> tuple[bytes, bool]:
    """
    Sửa lỗi 0x20 thay vì 0x00 trong PCAP global header + packet header.
    Không đụng payload.
    Trả về (repaired_bytes, changed_flag).
    """
    buf = bytearray(raw)
    if len(buf) < 24:
        return raw, False

    magic = bytes(buf[:4])
    if magic == b'\xd4\xc3\xb2\xa1':  # little endian
        endian = '<'
    elif magic == b'\xa1\xb2\xc3\xd4':  # big endian
        endian = '>'
    else:
        endian = '<'  # fallback

    changed = _fix_space_bytes(buf, 0, 24)

    off = 24
    while off + 16 <= len(buf):
        changed |= _fix_space_bytes(buf, off, off + 16)
        try:
            caplen = struct.unpack_from(endian + 'I', buf, off + 8)[0]
        except struct.error:
            break

        # Dừng nếu header hỏng không có chiều dài hợp lệ
        if caplen == 0 or off + 16 + caplen > len(buf):
            break

        off += 16 + caplen

    return bytes(buf), changed


def _open_pcap_like(raw_bytes: bytes):
    """Thử parse bằng pcap -> pcapng, trả về reader hoặc None."""
    for reader_cls in (dpkt.pcap.Reader, dpkt.pcapng.Reader):
        try:
            return reader_cls(io.BytesIO(raw_bytes))
        except (ValueError, dpkt.dpkt.NeedData):
            continue
    return None


def load_pcap(path: str, repair_space_nulls: bool = False):
    """
    Đọc file và parse bằng dpkt.

    repair_space_nulls=True ép sửa 0x20→0x00 ở PCAP headers (an toàn cho payload).

    Trả về (iterator (ts_wall, dpkt_ethernet_frame), header_fixed_flag).
    """
    with open(path, 'rb') as f:
        raw = f.read()

    repaired_raw, changed = repair_pcap_headers(raw)

    candidates = []
    if changed or repair_space_nulls:
        # Ưu tiên bản đã sửa header nếu phát hiện có 0x20.
        candidates.append((repaired_raw, True))
    candidates.append((raw, False))

    for data, fixed in candidates:
        reader = _open_pcap_like(data)
        if reader:
            # Gắn cờ để log ở main
            setattr(reader, '_header_repaired', fixed)
            return reader, fixed

    raise ValueError("Unsupported pcap/pcapng format or corrupted capture.")


def repair_packet_headers(pkt: bytes) -> bytes:
    """
    Fix 0x20 → 0x00 trong phần header của từng packet:
      - EtherType (802.1Q) và inner EtherType
      - IPv4 header + UDP header
      - Base RTP header (12 bytes)
    Payload H264 giữ nguyên.
    """
    b = bytearray(pkt)
    if len(b) < 14:
        return pkt

    etype = struct.unpack_from('>H', b, 12)[0]
    vlan = False

    # Sửa TPID 0x8120 -> 0x8100, EtherType 0x0820 -> 0x0800
    if etype == 0x8120:
        b[13] = 0x00
        etype = 0x8100
    elif etype == 0x0820:
        b[13] = 0x00
        etype = 0x0800

    if etype == 0x8100 and len(b) >= 18:
        vlan = True
        inner = struct.unpack_from('>H', b, 16)[0]
        if inner == 0x0820:
            b[17] = 0x00
            inner = 0x0800
        etype = inner

    if etype != 0x0800:
        return bytes(b)

    ip_off = 14 + (4 if vlan else 0)
    if len(b) <= ip_off:
        return bytes(b)

    ihl = (b[ip_off] & 0x0f) * 4
    if ihl < 20 or ip_off + ihl > len(b):
        ihl = 20 if ip_off + 20 <= len(b) else len(b) - ip_off

    udp_off = ip_off + ihl
    udp_end = udp_off + 8
    if udp_off >= len(b):
        return bytes(b)

    # RTP header ngay sau UDP header (12 bytes tối thiểu)
    rtp_end = min(len(b), udp_end + 12)

    # Chỉ sửa từ EtherType trở đi để không động vào MAC.
    for i in range(12, rtp_end):
        if b[i] == 0x20:
            b[i] = 0x00

    return bytes(b)


# ─────────────────────────────────────────────
#  STEP 2: Extract RTP streams
# ─────────────────────────────────────────────

def extract_rtp_streams(pcap_reader) -> dict:
    """
    Parse tất cả packets, extract RTP payload.
    Hỗ trợ 802.1Q VLAN.
    Trả về dict: ssrc → [(seq, rtp_ts, marker, payload_bytes)]
    """
    streams = defaultdict(list)

    for ts_wall, buf in pcap_reader:
        # Sửa 0x20 → 0x00 trong phần header (Ethernet→RTP header), không chạm payload.
        buf = repair_packet_headers(buf)
        try:
            eth = dpkt.ethernet.Ethernet(buf)

            # Handle 802.1Q VLAN
            if eth.type == 0x8100:
                vlan_bytes = bytes(buf[14:])
                inner_et = struct.unpack_from('>H', vlan_bytes, 2)[0]
                if inner_et != 0x0800:
                    continue
                ip = dpkt.ip.IP(vlan_bytes[4:])
            elif isinstance(eth.data, dpkt.ip.IP):
                ip = eth.data
            else:
                continue

            if not isinstance(ip.data, dpkt.udp.UDP):
                continue

            rtp_raw = bytes(ip.data.data)
            if len(rtp_raw) < 12:
                continue

            # Parse RTP header
            b0 = rtp_raw[0]
            if (b0 >> 6) != 2:
                continue

            has_ext = (b0 >> 4) & 1
            cc      = b0 & 0xf
            marker  = (rtp_raw[1] >> 7) & 1
            seq     = struct.unpack_from('>H', rtp_raw, 2)[0]
            rtp_ts  = struct.unpack_from('>I', rtp_raw, 4)[0]
            ssrc    = struct.unpack_from('>I', rtp_raw, 8)[0]

            off = 12 + cc * 4
            if has_ext and off + 4 <= len(rtp_raw):
                ext_len = struct.unpack_from('>H', rtp_raw, off + 2)[0]
                off += 4 + ext_len * 4

            payload = rtp_raw[off:]
            streams[ssrc].append((seq, rtp_ts, marker, payload))

        except Exception:
            continue

    return streams


def print_stream_info(streams: dict):
    print(f"\n  {'SSRC':<14} {'Pkts':>5}  {'KB':>8}  {'Duration':>10}")
    print(f"  {'-'*50}")
    for ssrc, pkts in sorted(streams.items(), key=lambda x: -len(x[1])):
        tss = [p[1] for p in pkts]
        dur = (max(tss) - min(tss)) / RTP_CLOCK if tss else 0
        kb  = sum(len(p[3]) for p in pkts) / 1024
        print(f"  {hex(ssrc):<14} {len(pkts):>5}  {kb:>8.1f}  {dur:>9.2f}s")


# ─────────────────────────────────────────────
#  STEP 3: Assemble H264 frames from RTP
# ─────────────────────────────────────────────

START_CODE = b'\x00\x00\x00\x01'


def assemble_h264_frames(rtp_packets: list) -> list:
    """
    Nhận list (seq, rtp_ts, marker, payload) theo thứ tự pcap arrival.
    Trả về list of (rtp_ts, annexb_bytes).

    Xử lý:
      - Single NAL (type 1-23)
      - STAP-A (type 24): nhiều NAL trong 1 packet
      - FU-A (type 28): NAL lớn được phân mảnh
    """
    # Group by RTP timestamp (= 1 video frame), giữ pcap arrival order
    frames_by_ts = defaultdict(list)
    for seq, rtp_ts, marker, payload in rtp_packets:
        frames_by_ts[rtp_ts].append((seq, payload))

    result = []

    for fts in sorted(frames_by_ts.keys()):
        pkts  = frames_by_ts[fts]
        frags = {}   # fu_nt → accumulated data
        nalus = []

        for seq, payload in pkts:
            if not payload:
                continue
            nal_type = payload[0] & 0x1f

            # ── Single NAL (1–23) ────────────────────────
            if 1 <= nal_type <= 23:
                nalus.append(payload)

            # ── STAP-A (24) ──────────────────────────────
            elif nal_type == 24:
                idx = 1
                while idx + 2 <= len(payload):
                    sz = struct.unpack_from('>H', payload, idx)[0]
                    if (sz == 0 or idx + 2 + sz > len(payload)) and payload[idx] == 0x20:
                        # File bị chuyển 0x00 → 0x20 ở byte độ dài NAL.
                        sz = struct.unpack('>H', b'\x00' + bytes([payload[idx + 1]]))[0]
                    idx += 2
                    if sz == 0 or idx + sz > len(payload):
                        break
                    nalus.append(payload[idx:idx + sz])
                    idx += sz

            # ── FU-A (28) ────────────────────────────────
            elif nal_type == 28:
                if len(payload) < 2:
                    continue
                fu_hdr  = payload[1]
                is_s    = bool(fu_hdr & 0x80)
                is_e    = bool(fu_hdr & 0x40)
                fu_nt   = fu_hdr & 0x1f
                nal_hdr = (payload[0] & 0xe0) | fu_nt

                if is_s:
                    frags[fu_nt] = bytes([nal_hdr]) + payload[2:]
                elif fu_nt in frags:
                    frags[fu_nt] += payload[2:]

                if is_e and fu_nt in frags:
                    nalus.append(frags.pop(fu_nt))
            # không flush orphan frags

        if nalus:
            result.append((fts, b''.join(START_CODE + n for n in nalus)))

    return result


# ─────────────────────────────────────────────
#  STEP 4: Decode H264 → YUV raw frames
# ─────────────────────────────────────────────

def decode_to_yuv(h264_path: str, fps: float, width: int, height: int) -> bytes:
    """Dùng ffmpeg decode H264 Annex B → raw YUV420p."""
    r = subprocess.run([
        'ffmpeg', '-y',
        '-flags', '+output_corrupt',
        '-max_error_rate', '1.0',
        '-r', f'{fps:.6f}',
        '-i', h264_path,
        '-f', 'rawvideo', '-pix_fmt', 'yuv420p', '-'
    ], capture_output=True)
    return r.stdout


def get_decoded_pts(h264_path: str, fps: float) -> list:
    """Lấy PTS values của các frames được decode thành công."""
    r = subprocess.run([
        'ffmpeg', '-y',
        '-flags', '+output_corrupt', '-max_error_rate', '1.0',
        '-r', f'{fps:.6f}', '-i', h264_path,
        '-vf', 'showinfo', '-f', 'null', '-'
    ], capture_output=True, text=True)
    pts_list = []
    for line in r.stderr.splitlines():
        m = re.search(r'pts_time:\s*([\d.]+)', line)
        if m:
            pts_list.append(float(m.group(1)))
    return pts_list


# ─────────────────────────────────────────────
#  STEP 5: Build complete YUV + encode MP4
# ─────────────────────────────────────────────

def build_complete_yuv(decoded_yuv_data: bytes, decoded_pts: list,
                       frame_list: list, t0: int,
                       width: int, height: int) -> bytes:
    """
    Map decoded frames về đúng RTP timestamp.
    Fill missing frames (H264 decode errors) bằng last good frame.
    Trả về complete YUV byte string với đủ 527 frames.
    """
    frame_size = width * height * 3 // 2
    decoded_frames = [
        decoded_yuv_data[i * frame_size:(i + 1) * frame_size]
        for i in range(len(decoded_yuv_data) // frame_size)
    ]

    # Map decoded frame idx → closest RTP frame idx
    rtp_pts_list = [(frame_list[i][0] - t0) / RTP_CLOCK for i in range(len(frame_list))]
    rtp_to_decoded = {}
    for di, dpts in enumerate(decoded_pts):
        if di >= len(decoded_frames):
            break
        best_rtp = min(range(len(frame_list)),
                       key=lambda j: abs(rtp_pts_list[j] - dpts))
        rtp_to_decoded[best_rtp] = di

    # Build complete sequence
    result = bytearray()
    last_good = decoded_frames[0] if decoded_frames else b'\x80' * frame_size
    filled = 0

    for rtp_idx in range(len(frame_list)):
        if rtp_idx in rtp_to_decoded:
            frame = decoded_frames[rtp_to_decoded[rtp_idx]]
            last_good = frame
        else:
            frame = last_good   # duplicate last good
            filled += 1
        result.extend(frame)

    return bytes(result), filled


def encode_mp4(yuv_path: str, output_path: str,
               fps: float, width: int, height: int):
    """Encode raw YUV → MP4."""
    return subprocess.run([
        'ffmpeg', '-y',
        '-f', 'rawvideo', '-pix_fmt', 'yuv420p',
        '-s', f'{width}x{height}',
        '-r', f'{fps:.6f}',
        '-i', yuv_path,
        '-c:v', 'libx264', '-preset', 'fast', '-crf', '18',
        '-color_range', '1',
        '-colorspace', 'smpte170m',
        '-color_primaries', 'smpte170m',
        '-color_trc', 'smpte170m',
        output_path
    ], capture_output=True, text=True)


# ─────────────────────────────────────────────
#  MAIN
# ─────────────────────────────────────────────

def main():
    args = sys.argv[1:]
    force_ssrc   = None
    stream_pref  = 'recv'  # recv = nhiều data hơn (downstream)
    repair_space_nulls = False

    if '--ssrc' in args:
        i = args.index('--ssrc')
        force_ssrc = int(args[i + 1], 16)
        args = [a for j, a in enumerate(args) if j not in (i, i + 1)]
    if '--stream' in args:
        i = args.index('--stream')
        stream_pref = args[i + 1]
        args = [a for j, a in enumerate(args) if j not in (i, i + 1)]
    if '--repair-space-nulls' in args:
        repair_space_nulls = True
        args = [a for a in args if a != '--repair-space-nulls']

    if not args:
        print("Usage: python3 decode_rtp_h264_v7.py <input.pcap|input.pcapng> [output.mp4]")
        print("       [--ssrc 0xXXXXXXXX] [--stream recv|send] [--repair-space-nulls]")
        sys.exit(1)

    input_pcap  = args[0]
    output_mp4  = args[1] if len(args) > 1 else 'output.mp4'
    output_h264 = output_mp4.replace('.mp4', '') + '.h264'
    output_yuv  = output_mp4.replace('.mp4', '') + '_raw.yuv'

    if not os.path.exists(input_pcap):
        print(f"[!] File not found: {input_pcap}")
        sys.exit(1)

    for tool in ('ffmpeg', 'ffprobe'):
        if shutil.which(tool) is None:
            print(f"[!] Required tool missing: {tool}. Please install ffmpeg/ffprobe.")
            sys.exit(1)

    # ── 1. Load pcap ─────────────────────────────────────
    mode_note = "header-space-fix=FORCED" if repair_space_nulls else "header-space-fix=AUTO"
    print(f"\n[1/6] Load pcap: {input_pcap} ({mode_note})")
    print(f"      ({os.path.getsize(input_pcap):,} bytes)")
    try:
        pcap, header_fixed = load_pcap(input_pcap, repair_space_nulls=repair_space_nulls)
    except ValueError as e:
        print(f"[!] {e}")
        sys.exit(1)
    if header_fixed:
        print("      Repaired 0x20 → 0x00 in PCAP headers (global + per-packet)")

    # ── 2. Extract RTP ───────────────────────────────────
    print("[2/6] Extract RTP streams (dpkt) ...")
    streams = extract_rtp_streams(pcap)
    if not streams:
        print("[!] No RTP streams found. Check capture format/VLAN/UDP payload.")
        sys.exit(1)
    print_stream_info(streams)

    payload_sizes = {s: sum(len(p[3]) for p in pkts) for s, pkts in streams.items()}
    non_empty_streams = {s: payload_sizes[s] for s in payload_sizes if payload_sizes[s] > 0}
    stream_pool = non_empty_streams if non_empty_streams else payload_sizes

    # Select target SSRC
    if force_ssrc:
        target = force_ssrc
    elif stream_pref == 'recv':
        target = max(stream_pool, key=lambda s: (stream_pool[s], len(streams[s])))
    else:
        target = min(stream_pool, key=lambda s: (stream_pool[s], len(streams[s])))

    print(f"\n      → Target SSRC: {hex(target)}")

    # ── 3. Assemble H264 frames ──────────────────────────
    print("\n[3/6] Assemble H264 frames from RTP ...")
    rtp_pkts   = streams[target]
    frame_list = assemble_h264_frames(rtp_pkts)

    if not frame_list:
        print("[!] No frames assembled!")
        sys.exit(1)

    t0  = frame_list[0][0]
    dur = (frame_list[-1][0] - t0) / RTP_CLOCK

    # Compute FPS from median RTP timestamp interval
    ts_vals  = [fts for fts, _ in frame_list]
    diffs    = sorted([ts_vals[i+1] - ts_vals[i]
                       for i in range(len(ts_vals)-1)
                       if ts_vals[i+1] > ts_vals[i]])
    fps      = RTP_CLOCK / diffs[len(diffs)//2] if diffs else 15.0

    # Get video dimensions from SPS
    width = 240; height = 320  # defaults
    for _, data in frame_list:
        pos = 0
        while pos < len(data) - 5:
            if data[pos:pos+4] == START_CODE:
                nt = data[pos+4] & 0x1f
                if nt == 7 and pos + 10 < len(data):
                    # SPS: skip decoding, use defaults or detect from ffprobe later
                    pass
            pos += 1
        break

    print(f"      Frames: {len(frame_list)}")
    print(f"      Duration: {dur:.2f}s")
    print(f"      FPS: {fps:.2f}")

    with open(output_h264, 'wb') as f:
        for _, data in frame_list:
            f.write(data)
    print(f"      H264 written: {output_h264} ({os.path.getsize(output_h264):,} bytes)")

    # ── 4. Decode H264 → YUV ────────────────────────────
    print("\n[4/6] Decode H264 → YUV (ffmpeg) ...")

    # Detect actual dimensions from the H264 file
    probe = subprocess.run([
        'ffprobe', '-v', 'error', '-show_entries', 'stream=width,height',
        '-of', 'default', output_h264
    ], capture_output=True, text=True)
    for line in probe.stdout.splitlines():
        if 'width=' in line:
            width = int(line.split('=')[1])
        if 'height=' in line:
            height = int(line.split('=')[1])

    if width <= 0 or height <= 0:
        # fallback an toàn nếu ffprobe không đọc được do SPS hỏng
        width, height = 240, 320

    print(f"      Video: {width}x{height}")

    yuv_data = decode_to_yuv(output_h264, fps, width, height)
    frame_size = width * height * 3 // 2
    decoded_count = len(yuv_data) // frame_size
    print(f"      Decoded: {decoded_count}/{len(frame_list)} frames")

    decoded_pts = get_decoded_pts(output_h264, fps)
    print(f"      PTS mapped: {len(decoded_pts)} frames")

    # ── 5. Build complete sequence ───────────────────────
    print("\n[5/6] Build complete frame sequence ...")
    complete_yuv, filled = build_complete_yuv(
        yuv_data, decoded_pts, frame_list, t0, width, height
    )
    total_frames = len(complete_yuv) // frame_size
    print(f"      Good frames:      {total_frames - filled}")
    print(f"      Duplicated frames: {filled}")
    print(f"      Total frames:     {total_frames}")

    with open(output_yuv, 'wb') as f:
        f.write(complete_yuv)

    # ── 6. Encode MP4 ────────────────────────────────────
    print(f"\n[6/6] Encode → {output_mp4} ...")
    result = encode_mp4(output_yuv, output_mp4, fps, width, height)

    if result.returncode == 0 and os.path.exists(output_mp4):
        fs = [l for l in result.stderr.splitlines() if 'frame=' in l and 'fps=' in l]
        if fs:
            print(f"      {fs[-1].strip()}")

        probe2 = subprocess.run([
            'ffprobe', '-v', 'error', '-show_entries',
            'format=duration:stream=nb_frames,width,height,r_frame_rate',
            '-of', 'default', output_mp4
        ], capture_output=True, text=True)

        print("\n  Output info:")
        for line in probe2.stdout.splitlines():
            if any(k in line for k in ('nb_frames', 'duration', 'width', 'height', 'frame_rate')):
                print(f"    {line.strip()}")

        # Cleanup temp YUV (large file)
        if os.path.exists(output_yuv):
            os.remove(output_yuv)
            print(f"    (temp YUV removed)")

        print(f"\n✓ Done: {output_mp4} ({os.path.getsize(output_mp4):,} bytes)")
    else:
        print(f"[!] Encode failed (rc={result.returncode})")
        print(result.stderr[-300:])
        print(f"    H264 raw: {output_h264}")


if __name__ == '__main__':
    main()

Writing main.py


In [3]:
# !python main.py /kaggle/input/datasets/ngtht71/vidcall/video_1_84332002263_10444-b7kd7i1u7fdq7e49rdguwtiumw.txt video_1_84332002263_10444.mp4
# !python main.py /kaggle/input/datasets/ngtht71/vidcall/video_2_84332002263_10416-f4k7jewk3p8etbnxqc5dhudiiw.txt video_2_84332002263_10416.mp4
# !python main.py /kaggle/input/datasets/ngtht71/vidcall/video_1_84862084735_10448-pkoftwezpidd8nquhidppofzga.txt video_1_84862084735_10448.mp4
# !python main.py /kaggle/input/datasets/ngtht71/vidcall/video_2_84862084735_10412-qmzzr3rymjdgtn19wt5mnu8pmy.txt video_2_84862084735_10412.mp4

# INFERENCE

In [4]:
import asyncio
import socket
import struct
import threading
import time
import os
import sys
import argparse
import queue
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field
from enum import Enum
from typing import Optional, Callable
 
import av
import cv2
import numpy as np
import mediapipe as mp
import dpkt
import socket
import time
import io
from fractions import Fraction

2026-05-14 04:24:21.177227: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778732661.605155      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778732661.716223      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778732662.665577      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778732662.665618      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778732662.665620      57 computation_placer.cc:177] computation placer alr

In [5]:
BaseOptions       = mp.tasks.BaseOptions
FaceLandmarker    = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOpt = mp.tasks.vision.FaceLandmarkerOptions
FaceLandmarkerRes = mp.tasks.vision.FaceLandmarkerResult
ImageSegmenter    = mp.tasks.vision.ImageSegmenter
ImageSegmenterOpt = mp.tasks.vision.ImageSegmenterOptions
VisionRunningMode = mp.tasks.vision.RunningMode
MPImage           = mp.Image
MPImageFormat     = mp.ImageFormat

## CONFIG

In [6]:
# CONFIG

class EffectType(Enum):
    FACE_AR    = "face_ar"
    BG_REMOVE  = "bg_remove"
    BG_REPLACE = "bg_replace"
    BG_BLUR    = "bg_blur"
 
 
@dataclass
class PipelineConfig:
    # Network
    listen_ip:   str = "0.0.0.0"
    listen_port: int = 5004
    dest_ip:     str = "127.0.0.1"
    dest_port:   int = 5006
    ssrc_in:     Optional[int] = None   # None = auto-detect từ traffic
 
    # Video
    fps:      float = 30
    width:    int   = 240
    height:   int   = 320
    bitrate:  int   = 500_000           # bps
 
    # Models
    face_model_path: str = "face_landmarker.task"
    seg_model_path:  str = "selfie_segmenter.tflite"
 
    # Effect
    effect:       EffectType = EffectType.BG_BLUR
    bg_image_path: str       = ""
    blur_ksize:   int        = 55
 
    # Timing
    jitter_buffer_ms: int = 80    # flush sau bao lâu nếu không có marker
    max_queue_size:   int = 10    # drop frame nếu hàng đợi đầy

## RTP RECEIVER

In [7]:
# 1. RTP RECEIVER — Jitter buffer + FU-A assembly
 
START_CODE = b'\x00\x00\x00\x01'
SEQ_MOD    = 65536
RTP_CLOCK  = 90000
 
 
# class JitterBuffer:
#     """
#     Sắp xếp lại RTP packets theo seq number, flush khi:
#       - Gặp marker bit = 1 (end of access unit)
#       - Hoặc timeout jitter_buffer_ms (tránh treo khi mất marker)
#     """
#     def __init__(self, jitter_ms: float = 80):
#         self._buf: dict[int, tuple] = {}   # seq → (rtp_ts, marker, payload)
#         self._anchor_seq: Optional[int] = None
#         self._last_flush_time = time.time()
#         self._jitter_s = jitter_ms / 1000.0
 
#     def push(self, seq: int, rtp_ts: int, marker: int, payload: bytes):
#         self._buf[seq] = (rtp_ts, marker, payload)
#         if self._anchor_seq is None:
#             self._anchor_seq = seq
 
#     def flush(self) -> list[tuple]:
#         """Trả về danh sách (seq, rtp_ts, marker, payload) đã sắp xếp."""
#         if not self._buf:
#             return []
#         now = time.time()
#         has_marker = any(v[1] == 1 for v in self._buf.values())
#         timeout    = (now - self._last_flush_time) > self._jitter_s
 
#         if not (has_marker or timeout):
#             return []
 
#         anchor = self._anchor_seq or min(self._buf.keys())
 
#         def seq_key(s):
#             d = (s - anchor) % SEQ_MOD
#             return d if d < SEQ_MOD // 2 else d - SEQ_MOD
 
#         sorted_seqs = sorted(self._buf.keys(), key=seq_key)
#         result = [(s, *self._buf[s]) for s in sorted_seqs]
#         self._buf.clear()
#         self._anchor_seq = None
#         self._last_flush_time = now
#         return result


class TimestampFrameAssembler:
    """
    Assemble RTP packets theo RTP timestamp.
    IMS stream ổn định hơn marker-bit.
    """

    def __init__(self):
        self.frames = defaultdict(list)
        self.current_ts = None

    def push(self, seq, rtp_ts, payload):
        self.frames[rtp_ts].append((seq, payload))

        if self.current_ts is None:
            self.current_ts = rtp_ts

        if rtp_ts != self.current_ts:
            old_ts = self.current_ts
            self.current_ts = rtp_ts

            pkts = sorted(self.frames.pop(old_ts), key=lambda x: x[0])
            return old_ts, pkts

        return None


 
class FuaAssembler:
    """RFC 6184 §5.8: FU-A fragment assembler per RTP timestamp."""
    def __init__(self):
        self._buf:     Optional[bytearray] = None
        self._fu_nt:   Optional[int]       = None
        self._exp_seq: Optional[int]       = None
 
    def reset(self):
        self._buf = self._fu_nt = self._exp_seq = None
 
    def feed(self, seq: int, payload: bytes) -> Optional[bytes]:
        if len(payload) < 2:
            self.reset()
            return None
 
        fi, fh  = payload[0], payload[1]
        is_s    = bool(fh & 0x80)
        is_e    = bool(fh & 0x40)
        fu_nt   = fh & 0x1f
        nal_hdr = (fi & 0xe0) | fu_nt
 
        if is_s:
            self._buf     = bytearray([nal_hdr]) + payload[2:]
            self._fu_nt   = fu_nt
            self._exp_seq = (seq + 1) % SEQ_MOD
 
        elif self._buf is not None:
            if seq != self._exp_seq or fu_nt != self._fu_nt:
                self.reset()
                return None
            self._buf    += payload[2:]
            self._exp_seq = (seq + 1) % SEQ_MOD
        else:
            return None
 
        if is_e and self._buf is not None:
            nal = bytes(self._buf)
            self.reset()
            return nal
        return None
 
 
def parse_rtp(data: bytes) -> Optional[tuple]:
    """Parse RTP header. Returns (seq, rtp_ts, ssrc, marker, pt, payload)."""
    if len(data) < 12:
        return None
    b0 = data[0]
    if (b0 >> 6) != 2:
        return None
    has_ext = (b0 >> 4) & 1
    cc      = b0 & 0x0f
    marker  = (data[1] >> 7) & 1
    pt      = data[1] & 0x7f
    seq     = struct.unpack_from('>H', data, 2)[0]
    rtp_ts  = struct.unpack_from('>I', data, 4)[0]
    ssrc    = struct.unpack_from('>I', data, 8)[0]
    hdr_end = 12 + cc * 4
    if has_ext:
        if hdr_end + 4 > len(data):
            return None
        ext_len  = struct.unpack_from('>H', data, hdr_end + 2)[0]
        hdr_end += 4 + ext_len * 4
    if hdr_end > len(data):
        return None
    return seq, rtp_ts, ssrc, marker, pt, data[hdr_end:]
 
 
# def assemble_annexb(sorted_pkts: list) -> Optional[bytes]:
#     """
#     Nhận danh sách (seq, rtp_ts, marker, payload) đã sắp xếp theo seq.
#     Trả về Annex-B bytes của 1 access unit, hoặc None nếu không có NAL nào.
#     """
#     nalus = []
#     fua   = FuaAssembler()
 
#     for seq, rtp_ts, marker, payload in sorted_pkts:
#         if not payload:
#             continue
#         nt      = payload[0] & 0x1f
#         new_nals: list[bytes] = []
 
#         if 1 <= nt <= 23:
#             # Single NAL unit
#             new_nals.append(payload)
 
#         elif nt == 24:
#             # STAP-A: multiple NALs, size fields may have 0x20 corruption
#             idx = 1
#             while idx + 2 <= len(payload):
#                 sz_raw = payload[idx: idx + 2]
#                 sz     = struct.unpack('>H', sz_raw)[0]
#                 # Fix size nếu high byte bị 0x20 corruption
#                 if sz > len(payload) and sz_raw[0] == 0x20:
#                     sz = struct.unpack('>H', bytes([0x00, sz_raw[1]]))[0]
#                 idx += 2
#                 if sz == 0 or idx + sz > len(payload):
#                     break
#                 new_nals.append(payload[idx: idx + sz])
#                 idx += sz
 
#         elif nt == 28:
#             # FU-A fragmented NAL
#             nal = fua.feed(seq, payload)
#             if nal:
#                 new_nals.append(nal)
 
#         nalus.extend(new_nals)
 
#     if not nalus:
#         return None
#     return b''.join(START_CODE + n for n in nalus)

def assemble_access_unit(pkts):
    """
    Assemble full H264 access unit từ RTP packets.
    """

    nalus = []

    fua_buffer = bytearray()
    assembling = False

    for seq, payload in pkts:

        if not payload:
            continue

        nal_type = payload[0] & 0x1F

        # Single NAL
        if 1 <= nal_type <= 23:
            nalus.append(payload)

        # STAP-A
        elif nal_type == 24:

            idx = 1

            while idx + 2 <= len(payload):

                size = struct.unpack(">H", payload[idx:idx+2])[0]
                idx += 2

                if idx + size > len(payload):
                    break

                nalus.append(payload[idx:idx+size])
                idx += size

        # FU-A
        elif nal_type == 28:

            fu_header = payload[1]

            start = fu_header & 0x80
            end   = fu_header & 0x40

            reconstructed_type = fu_header & 0x1F

            nri = payload[0] & 0x60

            nal_header = bytes([nri | reconstructed_type])

            if start:
                fua_buffer = bytearray()
                fua_buffer += nal_header
                fua_buffer += payload[2:]
                assembling = True

            elif assembling:
                fua_buffer += payload[2:]

            if end and assembling:
                nalus.append(bytes(fua_buffer))
                assembling = False

    if not nalus:
        return None

    return b''.join(
        b'\x00\x00\x00\x01' + n
        for n in nalus
    )
 
 
class RTPReceiver:
    """
    Lắng nghe UDP socket, reassemble H264 access units,
    đưa vào output_queue dưới dạng (rtp_ts, annexb_bytes).
    """
 
    def __init__(self, config: PipelineConfig, output_queue: queue.Queue):
        self._cfg     = config
        self._out_q   = output_queue
        self._sock    = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        self._sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        self._sock.bind((config.listen_ip, config.listen_port))
        self._sock.settimeout(0.5)
        self._running = False
        self._thread  = None
 
        # Per-SSRC state
        # self._jitter_bufs:  dict[int, JitterBuffer] = {}
        self._ssrc_target:  Optional[int]           = config.ssrc_in
        self._ssrc_pkt_cnt: dict[int, int]          = defaultdict(int)
        self._assembler = TimestampFrameAssembler()
 
    def start(self):
        self._running = True
        self._thread  = threading.Thread(target=self._recv_loop, daemon=True)
        self._thread.start()
        print(f"[RTPReceiver] Listening on {self._cfg.listen_ip}:{self._cfg.listen_port}")
 
    def stop(self):
        self._running = False
        if self._thread:
            self._thread.join(timeout=2)
        self._sock.close()
 
    # def _get_or_create_jbuf(self, ssrc: int) -> JitterBuffer:
    #     if ssrc not in self._jitter_bufs:
    #         self._jitter_bufs[ssrc] = JitterBuffer(self._cfg.jitter_buffer_ms)
    #     return self._jitter_bufs[ssrc]
 
    def _auto_select_ssrc(self) -> Optional[int]:
        """Chọn SSRC có nhiều packet nhất (stream chính)."""
        if not self._ssrc_pkt_cnt:
            return None
        return max(self._ssrc_pkt_cnt, key=lambda s: self._ssrc_pkt_cnt[s])
 
    def _recv_loop(self):
        """Main receive loop — chạy trong thread riêng."""
        while self._running:
            try:
                data, addr = self._sock.recvfrom(4096)
            except socket.timeout:
                # Flush timeout jitter buffers
                if self._ssrc_target:
                    jbuf   = self._get_or_create_jbuf(self._ssrc_target)
                    sorted_pkts = jbuf.flush()
                    if sorted_pkts:
                        self._emit(sorted_pkts)
                continue
            except Exception as e:
                if self._running:
                    print(f"[RTPReceiver] recv error: {e}")
                continue
 
            rtp = parse_rtp(data)
            if rtp is None:
                continue
 
            seq, rtp_ts, ssrc, marker, pt, payload = rtp
            self._ssrc_pkt_cnt[ssrc] += 1
 
            # Auto-detect SSRC sau 30 packet
            if self._ssrc_target is None:
                total = sum(self._ssrc_pkt_cnt.values())
                if total >= 30:
                    self._ssrc_target = self._auto_select_ssrc()
                    if self._ssrc_target:
                        print(f"[RTPReceiver] Auto-selected SSRC: 0x{self._ssrc_target:08x}")
 
            if self._ssrc_target and ssrc != self._ssrc_target:
                continue
 
            # jbuf = self._get_or_create_jbuf(ssrc)
            # jbuf.push(seq, rtp_ts, marker, payload)

            result = self._assembler.push(seq, rtp_ts, payload)
            if result:
                old_ts, pkts = result
                annexb = assemble_access_unit(pkts)
                if annexb:
                    self._emit(old_ts, annexb)
             
            # if marker:
            #     sorted_pkts = jbuf.flush()
            #     if sorted_pkts:
            #         self._emit(sorted_pkts)
 
    def _emit(self, sorted_pkts: list):
        """Assemble và đưa vào output queue."""
        rtp_ts  = sorted_pkts[0][1]
        # annexb  = assemble_annexb(sorted_pkts)
        if annexb is None:
            return
 
        try:
            self._out_q.put_nowait((rtp_ts, annexb))
        except queue.Full:
            # Drop oldest, push newest (real-time ưu tiên frame mới)
            try:
                self._out_q.get_nowait()
            except queue.Empty:
                pass
            try:
                self._out_q.put_nowait((rtp_ts, annexb))
            except queue.Full:
                pass

## H264 DECODER

In [8]:
# 2. H264 DECODER — PyAV, decode từng access unit
 
# class H264Decoder:
#     """
#     Decode H264 Annex-B bytes → numpy BGR frame.
#     Dùng PyAV, không ghi file tạm.
#     """
 
#     def __init__(self):
#         self._codec = av.CodecContext.create('h264', 'r')
#         # Không set width/height — auto-detect từ SPS
 
#     def decode(self, annexb: bytes) -> Optional[np.ndarray]:
#         """Trả về numpy array (H, W, 3) BGR hoặc None nếu không decode được."""
#         try:
#             pkt    = av.Packet(annexb)
#             frames = self._codec.decode(pkt)
#             for frame in frames:
#                 return frame.to_ndarray(format='bgr24')
#         except Exception as e:
#             # Suppress common errors từ corrupted frames
#             pass
#         return None
 
#     def close(self):
#         self._codec.close()


import subprocess

class ContinuousH264Decoder:

    def __init__(self, width=240, height=320):

        self.width = width
        self.height = height

        self.proc = subprocess.Popen(
            [
                "ffmpeg",
                "-loglevel", "quiet",
                "-fflags", "nobuffer",
                "-flags", "low_delay",

                "-f", "h264",
                "-i", "-",

                "-f", "rawvideo",
                "-pix_fmt", "bgr24",
                "-"
            ],
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            bufsize=10**8
        )

    def decode(self, annexb):

        self.proc.stdin.write(annexb)

        frame_size = self.width * self.height * 3

        raw = self.proc.stdout.read(frame_size)

        if len(raw) != frame_size:
            return None

        frame = np.frombuffer(raw, np.uint8)

        return frame.reshape(
            (self.height, self.width, 3)
        )

    def close(self):

        self.proc.stdin.close()
        self.proc.terminate()

## INFERENCE PIPELINE

In [9]:
# 3. INFERENCE PIPELINE — MediaPipe Tasks API (new)
#    FaceLandmarker + ImageSegmenter chạy LIVE_STREAM (callback-based)
 
class FaceLandmarkTrack:
    """
    MediaPipe Tasks FaceLandmarker — LIVE_STREAM mode.
    
    Thay đổi so với mp.solutions.face_mesh (deprecated):
      - DÙNG:       mp.tasks.vision.FaceLandmarker
      - Model:      .task bundle file (download riêng)
      - Input:      mp.Image thay vì numpy array trực tiếp
      - Timestamp:  bắt buộc phải cung cấp (ms), tăng đơn điệu
    """
 
    def __init__(self, model_path: str):
        self._result: Optional[FaceLandmarkerRes] = None
        self._lock   = threading.Lock()
 
        def _on_result(result: FaceLandmarkerRes,
                       output_image: MPImage,
                       timestamp_ms: int):
            with self._lock:
                self._result = result
 
        options = FaceLandmarkerOpt(
            base_options=BaseOptions(model_asset_path=model_path),
            running_mode=VisionRunningMode.LIVE_STREAM,
            num_faces=1,
            min_face_detection_confidence=0.5,
            min_face_presence_confidence=0.5,
            min_tracking_confidence=0.5,
            output_face_blendshapes=False,    # False để tiết kiệm latency
            output_facial_transformation_matrixes=True,
            result_callback=_on_result,
        )
        self._landmarker = FaceLandmarker.create_from_options(options)
        self._ts_ms      = 0    # monotonically increasing timestamp
 
    def infer_async(self, frame_bgr: np.ndarray):
        """
        Gửi frame đến landmarker bất đồng bộ.
        Kết quả nhận qua callback _on_result.
        """
        rgb      = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        mp_image = MPImage(image_format=MPImageFormat.SRGB, data=rgb)
        self._ts_ms += 1   # tăng mỗi frame, không cần đúng wall-clock
        self._landmarker.detect_async(mp_image, self._ts_ms)
 
    def get_latest(self) -> Optional[FaceLandmarkerRes]:
        with self._lock:
            return self._result
 
    def close(self):
        self._landmarker.close()
 
 
class SegmentationTrack:
    """
    MediaPipe Tasks ImageSegmenter — LIVE_STREAM mode.
    
    Thay đổi so với mp.solutions.selfie_segmentation (deprecated):
      - KHÔNG dùng: mp.solutions.selfie_segmentation.SelfieSegmentation
      - DÙNG:       mp.tasks.vision.ImageSegmenter
      - Model:      selfie_segmenter.tflite hoặc selfie_multiclass_256x256.tflite
      - Output:     confidence_masks (float32) hoặc category_mask (uint8)
 
    Model selfie_segmenter.tflite:
      - 2 categories: background (0), person (1)
      - Dùng output_confidence_masks=True → masks[1] = person confidence
    """
 
    def __init__(self, model_path: str):
        self._mask: Optional[np.ndarray] = None
        self._lock = threading.Lock()
 
        def _on_result(result, output_image, timestamp_ms):
            try:
                if result.confidence_masks:
                    masks = result.confidence_masks
        
                    if len(masks) == 1:
                        person_mask = masks[0].numpy_view()
                    elif len(masks) > 1:
                        person_mask = masks[1].numpy_view()
                    else:
                        return
        
                    with self._lock:
                        self._mask = (person_mask * 255).astype(np.uint8)
        
                elif result.category_mask is not None:
                    cat = result.category_mask.numpy_view()
                    with self._lock:
                        self._mask = np.where(cat == 1, 255, 0).astype(np.uint8)
        
            except Exception as e:
                # tránh crash thread callback
                pass
 
        options = ImageSegmenterOpt(
            base_options=BaseOptions(model_asset_path=model_path),
            running_mode=VisionRunningMode.LIVE_STREAM,
            output_confidence_masks=True,
            output_category_mask=False,
            result_callback=_on_result,
        )
        self._segmenter = ImageSegmenter.create_from_options(options)
        self._ts_ms     = 0
        self._prev_mask: Optional[np.ndarray] = None
 
    def infer_async(self, frame_bgr: np.ndarray):
        rgb      = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        mp_image = MPImage(image_format=MPImageFormat.SRGB, data=rgb)
        self._ts_ms += 1
        self._segmenter.segment_async(mp_image, self._ts_ms)
 
    def get_latest_mask(self) -> Optional[np.ndarray]:
        """Trả về mask uint8 (H,W), 255=người, 0=background."""
        with self._lock:
            mask = self._mask
        if mask is None:
            return None
        # Resize về kích thước gốc nếu cần (model output 256x256)
        return mask
 
    def close(self):
        self._segmenter.close()
 
 
class InferencePipeline:
    """
    Gọi FaceLandmarkTrack và SegmentationTrack song song.
    Dùng LIVE_STREAM mode nên cả hai non-blocking.
    """
 
    def __init__(self, config: PipelineConfig):
        self._cfg = config
        self._face_track: Optional[FaceLandmarkTrack] = None
        self._seg_track:  Optional[SegmentationTrack] = None
 
        effect = config.effect
        needs_face = (effect == EffectType.FACE_AR)
        needs_seg  = (effect in (EffectType.BG_REMOVE,
                                 EffectType.BG_REPLACE,
                                 EffectType.BG_BLUR))
 
        if needs_face:
            if not os.path.exists(config.face_model_path):
                print(f"[WARNING] Face model not found: {config.face_model_path}")
                print("  Download: wget https://storage.googleapis.com/mediapipe-models/"
                      "face_landmarker/face_landmarker/float16/1/face_landmarker.task")
            else:
                self._face_track = FaceLandmarkTrack(config.face_model_path)
                print(f"[Inference] FaceLandmarker loaded: {config.face_model_path}")
 
        if needs_seg:
            if not os.path.exists(config.seg_model_path):
                print(f"[WARNING] Seg model not found: {config.seg_model_path}")
                print("  Download: wget https://storage.googleapis.com/mediapipe-models/"
                      "image_segmenter/selfie_segmenter/float16/latest/selfie_segmenter.tflite")
            else:
                self._seg_track = SegmentationTrack(config.seg_model_path)
                print(f"[Inference] ImageSegmenter loaded: {config.seg_model_path}")
 
    def infer(self, frame_bgr: np.ndarray):
        """
        Gửi frame đến cả hai track KHÔNG ĐỒNG BỘ.
        Kết quả được cập nhật qua callback vào _result của mỗi track.
        Không block → overhead ~0.1 ms.
        """
        if self._face_track:
            self._face_track.infer_async(frame_bgr)
        if self._seg_track:
            self._seg_track.infer_async(frame_bgr)
 
    def get_face_result(self) -> Optional[FaceLandmarkerRes]:
        if self._face_track:
            return self._face_track.get_latest()
        return None
 
    def get_seg_mask(self, frame_shape: tuple) -> Optional[np.ndarray]:
        """Trả về mask đã resize về kích thước frame gốc."""
        if self._seg_track is None:
            return None
        mask = self._seg_track.get_latest_mask()
        if mask is None:
            return None
        h, w = frame_shape[:2]
        if mask.shape != (h, w):
            mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_LINEAR)
        return mask
 
    def close(self):
        if self._face_track:
            self._face_track.close()
        if self._seg_track:
            self._seg_track.close()

## EFFECT

In [10]:
# ═══════════════════════════════════════════════════════════════════════════
# 4. EFFECT COMPOSITOR
# ═══════════════════════════════════════════════════════════════════════════
 
class EffectCompositor:
    """
    Áp dụng video effect vào frame BGR dựa trên kết quả inference.
    """
 
    def __init__(self, config: PipelineConfig):
        self._cfg     = config
        self._bg_img: Optional[np.ndarray] = None
 
        if config.effect == EffectType.BG_REPLACE and config.bg_image_path:
            bg = cv2.imread(config.bg_image_path)
            if bg is not None:
                self._bg_img = bg
                print(f"[Compositor] Background image loaded: {config.bg_image_path}")
            else:
                print(f"[Compositor] WARNING: Cannot load bg image: {config.bg_image_path}")
 
    def apply(self,
              frame:      np.ndarray,
              face_res:   Optional[FaceLandmarkerRes],
              seg_mask:   Optional[np.ndarray]) -> np.ndarray:
        """
        Áp dụng effect phù hợp. Fallback: trả về frame gốc nếu thiếu input.
        """
        effect = self._cfg.effect
 
        if effect == EffectType.FACE_AR:
            if face_res is not None:
                return self._apply_face_ar(frame, face_res)
 
        elif effect == EffectType.BG_REMOVE:
            if seg_mask is not None:
                return self._remove_background(frame, seg_mask)
 
        elif effect == EffectType.BG_REPLACE:
            if seg_mask is not None and self._bg_img is not None:
                return self._replace_background(frame, seg_mask, self._bg_img)
            elif seg_mask is not None:
                return self._remove_background(frame, seg_mask)
 
        elif effect == EffectType.BG_BLUR:
            if seg_mask is not None:
                return self._blur_background(frame, seg_mask)
 
        return frame   # fallback: pass-through
 
    # ── Face AR effect ──────────────────────────────────────────────────────
 
    def _apply_face_ar(self,
                       frame:    np.ndarray,
                       face_res: FaceLandmarkerRes) -> np.ndarray:
        """
        Ví dụ: vẽ face mesh wireframe lên frame.
        Thay bằng logic AR thực tế (sticker, warp, virtual avatar).
        """
        if not face_res.face_landmarks:
            return frame
 
        out  = frame.copy()
        h, w = frame.shape[:2]
 
        for face_lms in face_res.face_landmarks:
            for lm in face_lms:
                x = int(lm.x * w)
                y = int(lm.y * h)
                cv2.circle(out, (x, y), 1, (0, 255, 0), -1)
 
        return out
 
    # ── Background remove ────────────────────────────────────────────────────
 
    def _remove_background(self,
                            frame: np.ndarray,
                            mask:  np.ndarray) -> np.ndarray:
        """Xóa background → nền đen. Mask 255=người, 0=bg."""
        mask_soft = self._soften_mask(mask, frame.shape[:2])
        mask_3    = np.stack([mask_soft] * 3, axis=2)
        return (frame * mask_3).astype(np.uint8)
 
    # ── Background replace ────────────────────────────────────────────────────
 
    def _replace_background(self,
                              frame:  np.ndarray,
                              mask:   np.ndarray,
                              bg_img: np.ndarray) -> np.ndarray:
        """Thay nền bằng ảnh tùy chọn."""
        h, w  = frame.shape[:2]
        bg    = cv2.resize(bg_img, (w, h))
 
        mask_f  = self._soften_mask(mask, (h, w))
        mask_3  = np.stack([mask_f] * 3, axis=2)
        fg_part = frame * mask_3
        bg_part = bg    * (1.0 - mask_3)
        return (fg_part + bg_part).astype(np.uint8)
 
    # ── Background blur ──────────────────────────────────────────────────────
 
    def _blur_background(self,
                          frame: np.ndarray,
                          mask:  np.ndarray) -> np.ndarray:
        """Blur nền, giữ nguyên foreground (người)."""
        ksize  = self._cfg.blur_ksize | 1   # phải lẻ
        blurred = cv2.GaussianBlur(frame, (ksize, ksize), 0)
 
        mask_f  = self._soften_mask(mask, frame.shape[:2])
        mask_3  = np.stack([mask_f] * 3, axis=2)
        fg_part = frame   * mask_3
        bg_part = blurred * (1.0 - mask_3)
        return (fg_part + bg_part).astype(np.uint8)
 
    # ── Helpers ──────────────────────────────────────────────────────────────
 
    @staticmethod
    def _soften_mask(mask: np.ndarray,
                     target_shape: tuple,
                     blur_ksize: int = 15) -> np.ndarray:
        """
        Resize mask → target, Gaussian blur viền để blend mềm (feathering).
        Trả về float32 [0.0, 1.0].
        """
        h, w = target_shape
        if mask.shape != (h, w):
            mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_LINEAR)
        mask_f  = mask.astype(np.float32) / 255.0
        # Feathering viền mask
        mask_blur = cv2.GaussianBlur(mask_f, (blur_ksize, blur_ksize), 0)
        return np.clip(mask_blur, 0.0, 1.0)

## H264 ENCODER

In [11]:
# ═══════════════════════════════════════════════════════════════════════════
# 5. H264 ENCODER — PyAV, zerolatency
# ═══════════════════════════════════════════════════════════════════════════
 
class H264Encoder:
    """
    Encode numpy BGR frame → H264 Annex-B bytes.
    Dùng zerolatency tune để giảm latency.
    """
 
    def __init__(self, config: PipelineConfig):
        self._cfg = config
        self._ctx = av.CodecContext.create('libx264', 'w')
        self._ctx.width        = config.width
        self._ctx.height       = config.height
        self._ctx.framerate    = config.fps
        self._ctx.bit_rate     = config.bitrate
        self._ctx.pix_fmt      = 'yuv420p'
        self._ctx.options      = {
            'tune':    'zerolatency',   # không có B-frame delay
            'preset':  'ultrafast',     # tốc độ > chất lượng
            'profile': 'baseline',      # IMS thường dùng baseline
        }
        self._frame_count = 0
 
    def encode(self, frame_bgr: np.ndarray) -> list[bytes]:
        """Trả về list Annex-B packet bytes (thường 1 packet/frame)."""
        h, w = frame_bgr.shape[:2]
 
        # Auto-resize nếu frame size thay đổi
        if w != self._ctx.width or h != self._ctx.height:
            frame_bgr = cv2.resize(frame_bgr, (self._ctx.width, self._ctx.height))
 
        av_frame           = av.VideoFrame.from_ndarray(frame_bgr, format='bgr24')
        av_frame.pts       = self._frame_count
        av_frame.time_base = Fraction(1, int(self._cfg.fps))
        self._frame_count += 1
 
        packets = self._ctx.encode(av_frame)
        return [bytes(p) for p in packets]
 
    def flush(self) -> list[bytes]:
        """Flush encoder buffer."""
        return [bytes(p) for p in self._ctx.encode(None)]
 
    def close(self):
        self.flush()
        # self._ctx.close()
        self._ctx = None

## RTP PACKET

In [12]:
# ═══════════════════════════════════════════════════════════════════════════
# 6. RTP PACKETIZER — RFC 6184 FU-A fragmentation
# ═══════════════════════════════════════════════════════════════════════════
 
class RTPPacketizer:
    """
    Nhận H264 Annex-B, đóng gói thành RTP packets và gửi UDP.
    Hỗ trợ FU-A fragmentation cho NAL unit lớn hơn MTU.
    """
 
    MTU_PAYLOAD = 1260   # safe cho IMS/LTE với RTP+UDP+IP+ETH overhead
 
    def __init__(self, config: PipelineConfig):
        self._dest    = (config.dest_ip, config.dest_port)
        self._ssrc    = os.getpid() & 0xFFFFFFFF   # dùng PID làm SSRC
        self._pt      = 114                         # dynamic payload type H264
        self._seq     = 0
        self._rtp_ts  = 0
        self._ts_incr = int(RTP_CLOCK / config.fps)
        self._sock    = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        print(f"[Packetizer] Sending to {config.dest_ip}:{config.dest_port}, SSRC=0x{self._ssrc:08x}")
 
    def send_annexb(self, annexb: bytes):
        """Tách Annex-B thành NAL units, đóng gói RTP, gửi đi."""
        nals = self._split_annexb(annexb)
        if not nals:
            return
 
        self._rtp_ts += self._ts_incr
 
        for i, nal in enumerate(nals):
            is_last_nal = (i == len(nals) - 1)
 
            if len(nal) <= self.MTU_PAYLOAD:
                # Single NAL unit packet
                marker = 1 if is_last_nal else 0
                self._send_rtp_packet(nal, marker)
            else:
                # FU-A fragmentation
                self._send_fua(nal, is_last_nal)
 
    def close(self):
        self._sock.close()
 
    # ── Internals ─────────────────────────────────────────────────────────
 
    @staticmethod
    def _split_annexb(annexb: bytes) -> list[bytes]:
        """Tách Annex-B stream thành list NAL unit bytes."""
        nals  = []
        i     = 0
        start = -1
 
        while i < len(annexb) - 3:
            if annexb[i:i+4] == b'\x00\x00\x00\x01':
                if start >= 0:
                    nals.append(annexb[start:i])
                start  = i + 4
                i     += 4
            elif annexb[i:i+3] == b'\x00\x00\x01':
                if start >= 0:
                    nals.append(annexb[start:i])
                start  = i + 3
                i     += 3
            else:
                i += 1
 
        if start >= 0 and start < len(annexb):
            nals.append(annexb[start:])
 
        return [n for n in nals if n]
 
    def _send_fua(self, nal: bytes, is_last_nal: bool):
        """Fragmented NAL → FU-A packets."""
        nal_hdr  = nal[0]
        payload  = nal[1:]
        chunk_sz = self.MTU_PAYLOAD - 2   # 2 bytes FU indicator + FU header
 
        chunks = [payload[i: i + chunk_sz]
                  for i in range(0, len(payload), chunk_sz)]
 
        for j, chunk in enumerate(chunks):
            is_start = (j == 0)
            is_end   = (j == len(chunks) - 1)
            marker   = 1 if (is_end and is_last_nal) else 0
 
            fu_ind = (nal_hdr & 0xe0) | 28          # NRI from original + type=28
            fu_hdr = ((is_start << 7) | (is_end << 6) | (nal_hdr & 0x1f))
            rtp_payload = bytes([fu_ind, fu_hdr]) + chunk
            self._send_rtp_packet(rtp_payload, marker)
 
    def _send_rtp_packet(self, payload: bytes, marker: int):
        """Đóng gói RTP header + payload, gửi UDP."""
        rtp_hdr = struct.pack(
            '>BBHII',
            0x80,                           # V=2, P=0, X=0, CC=0
            (marker << 7) | self._pt,
            self._seq & 0xFFFF,
            self._rtp_ts,
            self._ssrc,
        )
        self._sock.sendto(rtp_hdr + payload, self._dest)
        self._seq += 1

## SAVE OUTPUT

In [13]:
class VideoRecorder:
    def __init__(self,
                 output_path="output.mp4",
                 fps=30,
                 width=640,
                 height=480):

        self.output_path = output_path
        self.fps = fps
        self.width = width
        self.height = height

        fourcc = cv2.VideoWriter_fourcc(*'mp4v')

        self.writer = cv2.VideoWriter(
            output_path,
            fourcc,
            fps,
            (width, height)
        )

        print(f"[Recorder] Writing video: {output_path}")

    def write(self, frame):
        if frame is None:
            return

        h, w = frame.shape[:2]

        if (w, h) != (self.width, self.height):
            frame = cv2.resize(frame, (self.width, self.height))
    
        self.writer.write(frame)

    def close(self):
        self.writer.release()
        print(f"[Recorder] Saved: {self.output_path}")

## PIPELINE

In [14]:
# ═══════════════════════════════════════════════════════════════════════════
# 7. PIPELINE ORCHESTRATOR — kết nối tất cả module
# ═══════════════════════════════════════════════════════════════════════════
 
class VideoPipeline:
    """
    Orchestrator chính: kết nối Receiver → Decoder → Inference → Compositor
    → Encoder → Packetizer.
 
    Latency budget per frame (target total ≤ 100ms):
      RTP recv + jitter:   50 ms   (configurable)
      H264 decode:          8 ms
      Inference (async):   10 ms   (LIVE_STREAM, non-blocking send)
      Compositor:           5 ms
      H264 encode:         15 ms
      RTP send:             2 ms
    """
 
    def __init__(self, config: PipelineConfig):
        self._cfg    = config
        self._rtp_q  = queue.Queue(maxsize=config.max_queue_size)
 
        self._receiver     = RTPReceiver(config, self._rtp_q)
        # self._decoder      = H264Decoder()
        self._decoder      = ContinuousH264Decoder(width=config.width,height=config.height)
        self._inference    = InferencePipeline(config)
        self._compositor   = EffectCompositor(config)
        self._encoder      = H264Encoder(config)
        self._packetizer   = RTPPacketizer(config)
        self._latency = LatencyTracker()

        self._recorder = VideoRecorder(
            output_path="output_processed.mp4",
            fps=config.fps,
            width=config.width,
            height=config.height
        )
         
        self._running      = False
        self._process_thread = None
        self._frame_count  = 0
        self._drop_count   = 0
        self._t_start      = 0.0
 
    def start(self):
        self._running  = True
        self._t_start  = time.time()
        self._receiver.start()
 
        self._process_thread = threading.Thread(
            target=self._process_loop, daemon=True
        )
        self._process_thread.start()
        print(f"[Pipeline] Started. Effect: {self._cfg.effect.value}")
 
    def stop(self):
        self._running = False
        self._receiver.stop()
        if self._process_thread:
            self._process_thread.join(timeout=3)
        self._encoder.close()
        self._inference.close()
        self._packetizer.close()

        # self._recorder.close()
 
        elapsed = time.time() - self._t_start
        fps     = self._frame_count / elapsed if elapsed > 0 else 0
        print(f"\n[Pipeline] Stopped. Processed {self._frame_count} frames "
              f"({fps:.1f} fps), dropped {self._drop_count}")

        self._latency.report()
 
    def _process_loop(self):
        """Main processing loop — lấy từ queue, xử lý, gửi đi."""
        prev_frame: Optional[np.ndarray] = None
 
        # while self._running:
        #     try:
        #         rtp_ts, annexb = self._rtp_q.get(timeout=0.5)
        #     except queue.Empty:
        #         continue
 
        #     # t0 = time.perf_counter()

        #     t0 = self._latency.start()
 
        #     # ── Decode ────────────────────────────────────────────────────
        #     frame = self._decoder.decode(annexb)
        #     if frame is None:
        #         if prev_frame is not None:
        #             frame = prev_frame   # duplicate last good frame
        #             self._drop_count += 1
        #         else:
        #             continue   # không có gì để gửi
 
        #     # Update config dimensions từ actual frame size
        #     h, w = frame.shape[:2]
        #     if (self._cfg.width != w or self._cfg.height != h):
        #         self._cfg.width  = w
        #         self._cfg.height = h
 
        #     # ── Inference (non-blocking send) ─────────────────────────────
        #     self._inference.infer(frame)
 
        #     # ── Lấy kết quả inference của frame TRƯỚC (pipeline latency 1) ─
        #     # LIVE_STREAM callback async → kết quả N-1 frame ready khi N đang xử lý
        #     face_res  = self._inference.get_face_result()
        #     seg_mask  = self._inference.get_seg_mask(frame.shape)
 
        #     # ── Compositor ────────────────────────────────────────────────
        #     processed = self._compositor.apply(frame, face_res, seg_mask)

        #     # self._recorder.write(processed)
 
        #     # ── Encode ───────────────────────────────────────────────────
        #     packets = self._encoder.encode(processed)
        #     for pkt in packets:
        #         self._packetizer.send_annexb(pkt)
 
        #     prev_frame    = frame
        #     self._frame_count += 1
 
        #     # t1      = time.perf_counter()
        #     # elapsed = (t1 - t0) * 1000
        #     elapsed = self._latency.end(t0)

        #     cv2.putText(
        #         processed,
        #         f"Latency: {elapsed:.1f} ms",
        #         (20, 40),
        #         cv2.FONT_HERSHEY_SIMPLEX,
        #         1,
        #         (0, 255, 0),
        #         2
        #     )
            
        #     cv2.putText(
        #         processed,
        #         f"Frames: {self._frame_count}",
        #         (20, 80),
        #         cv2.FONT_HERSHEY_SIMPLEX,
        #         1,
        #         (0, 255, 0),
        #         2
        #     )
        #     compare = np.hstack([frame, processed])
        #     # save video output
        #     self._recorder.write(compare)
 
        #     # Log mỗi 30 frame
        #     if self._frame_count % 30 == 0:
        #         fps = self._frame_count / (time.time() - self._t_start)
        #         print(f"[Pipeline] frames={self._frame_count} "
        #               f"fps={fps:.1f} latency={elapsed:.1f}ms "
        #               f"dropped={self._drop_count} "
        #               f"queue={self._rtp_q.qsize()}")

        while self._running:

            total_t0 = time.perf_counter()
        
            rtp_ts, annexb = self._rtp_q.get()
        
            # decode
            t0 = time.perf_counter()
        
            frame = self._decoder.decode(annexb)
        
            decode_ms = (
                time.perf_counter() - t0
            ) * 1000
        
            self._latency.decode.append(decode_ms)
        
            if frame is None:
                continue
        
            # inference
            t0 = time.perf_counter()
        
            self._inference.infer(frame)
        
            seg_mask = self._inference.get_seg_mask(
                frame.shape
            )
        
            infer_ms = (
                time.perf_counter() - t0
            ) * 1000
        
            self._latency.infer.append(infer_ms)
        
            # compositor
            processed = self._compositor.apply(
                frame,
                None,
                seg_mask
            )
        
            # encode
            t0 = time.perf_counter()
        
            packets = self._encoder.encode(processed)
        
            encode_ms = (
                time.perf_counter() - t0
            ) * 1000
        
            self._latency.encode.append(encode_ms)
        
            for pkt in packets:
                self._packetizer.send_annexb(pkt)
        
            total_ms = (
                time.perf_counter() - total_t0
            ) * 1000
        
            self._latency.total.append(total_ms)

            cv2.putText(
                processed,
                f"Latency: {elapsed:.1f} ms",
                (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0, 255, 0),
                2
            )
            
            cv2.putText(
                processed,
                f"Frames: {self._frame_count}",
                (20, 80),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0, 255, 0),
                2
            )
            self._recorder.write(processed)

# RUN

In [15]:
class PcapReplayer:
    def __init__(self, pcap_path, dest_ip, dest_port,
                 ssrc_filter=None, speed=1.0):
        self._pcap_path = pcap_path
        self._dest = (dest_ip, dest_port)
        self._ssrc_filter = ssrc_filter
        self._speed = speed

        self._sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        self._running = False
        self._thread = None

    def start(self):
        self._running = True
        self._thread = threading.Thread(target=self._replay_loop, daemon=True)
        self._thread.start()
        print(f"[PCAP] Replaying {self._pcap_path}")

    def stop(self):
        self._running = False
        if self._thread:
            self._thread.join(timeout=2)

    def _replay_loop(self):
        prev_ts = None

        with open(self._pcap_path, "rb") as f:
            pcap = dpkt.pcap.Reader(f)

            for ts, buf in pcap:
                if not self._running:
                    break

                try:
                    eth = dpkt.ethernet.Ethernet(buf)
                    ip  = eth.data
                    if not isinstance(ip, dpkt.ip.IP):
                        continue
                    udp = ip.data
                    if not isinstance(udp, dpkt.udp.UDP):
                        continue

                    rtp = parse_rtp(udp.data)
                    if rtp is None:
                        continue

                    seq, rtp_ts, ssrc, marker, pt, payload = rtp

                    if self._ssrc_filter and ssrc != self._ssrc_filter:
                        continue

                    # giữ timing thật
                    if prev_ts is not None:
                        delay = (ts - prev_ts) / self._speed
                        if delay > 0:
                            time.sleep(delay)
                    prev_ts = ts

                    self._sock.sendto(udp.data, self._dest)

                except Exception:
                    continue


# ═══════════════════════════════════════════════════════════════════════════
# LATENCY TRACKER (NEW)
# ═══════════════════════════════════════════════════════════════════════════

# class LatencyTracker:
#     def __init__(self):
#         self.times = []

#     def start(self):
#         return time.perf_counter()

#     def end(self, t0):
#         t1 = time.perf_counter()
#         latency_ms = (t1 - t0) * 1000
#         self.times.append(latency_ms)
#         return latency_ms

#     def report(self):
#         if not self.times:
#             return

#         arr = np.array(self.times)
#         print("\n=== LATENCY REPORT ===")
#         print(f"Frames: {len(arr)}")
#         print(f"Avg: {arr.mean():.2f} ms")
#         print(f"P50: {np.percentile(arr,50):.2f} ms")
#         print(f"P90: {np.percentile(arr,90):.2f} ms")
#         print(f"P99: {np.percentile(arr,99):.2f} ms")
#         print(f"Max: {arr.max():.2f} ms")



class StageLatencyTracker:

    def __init__(self):

        self.decode = []
        self.infer = []
        self.encode = []
        self.total = []

    def add(self, arr, val):
        arr.append(val)

    def report(self):

        def stat(x):
            return np.mean(x), np.percentile(x,90)

        print("\n===== LATENCY =====")

        for name, arr in [
            ("decode", self.decode),
            ("infer", self.infer),
            ("encode", self.encode),
            ("total", self.total),
        ]:

            if arr:
                mean, p90 = stat(arr)

                print(
                    f"{name}: "
                    f"avg={mean:.2f}ms "
                    f"p90={p90:.2f}ms"
                )

In [16]:
# ── Config ────────────────────────────────────────────
PCAP_PATH         = '/kaggle/input/datasets/ngtht71/vidcall/video_1_84332002263_10444-b7kd7i1u7fdq7e49rdguwtiumw.txt'
FACE_MODEL_PATH   = 'face_landmarker.task'
SEG_MODEL_PATH    = 'selfie_segmenter.tflite'
TARGET_FPS        = 30
VIDEO_BITRATE     = 500_000
BENCHMARK_FRAMES  = 200   # số frame dùng để benchmark (tăng cho kết quả ổn hơn)

START_CODE = b'\x00\x00\x00\x01'
SEQ_MOD    = 65536
RTP_CLOCK  = 90000

print(f"\n[Config] PCAP: {PCAP_PATH}")
print(f"[Config] Face model: {FACE_MODEL_PATH}")
print(f"[Config] Seg model:  {SEG_MODEL_PATH}")
print(f"[Config] Benchmark frames: {BENCHMARK_FRAMES}")


[Config] PCAP: /kaggle/input/datasets/ngtht71/vidcall/video_1_84332002263_10444-b7kd7i1u7fdq7e49rdguwtiumw.txt
[Config] Face model: face_landmarker.task
[Config] Seg model:  selfie_segmenter.tflite
[Config] Benchmark frames: 200


In [17]:
t_stage1_start = time.perf_counter()
 
if not os.path.exists(PCAP_PATH):
    raise FileNotFoundError(
        f'PCAP not found: {PCAP_PATH}\\n'
        f'Copy your trace file here or adjust PCAP_PATH above.'
    )
 
filesize = os.path.getsize(PCAP_PATH)
print(f'Loading {PCAP_PATH} ({filesize/1e6:.2f} MB)...')
 
t_read = time.perf_counter()
with open(PCAP_PATH, 'rb') as f:
    raw = f.read()
t_read_end = time.perf_counter()
 
# Count and fix ALL 0x20 → 0x00 corruption
n_corrupted = sum(1 for b in raw if b == 0x20)
t_fix = time.perf_counter()
fixed_raw = bytes(0x00 if b == 0x20 else b for b in raw)
t_fix_end = time.perf_counter()
 
print(f'  File read:        {(t_read_end-t_read)*1000:.1f} ms')
print(f'  0x20 bytes found: {n_corrupted:,} ({100*n_corrupted/len(raw):.2f}% of file)')
print(f'  Fix (0x20→0x00):  {(t_fix_end-t_fix)*1000:.1f} ms')
 
t_parse = time.perf_counter()
pcap_reader = dpkt.pcap.Reader(io.BytesIO(fixed_raw))
raw_packets = list(pcap_reader)
t_parse_end = time.perf_counter()
 
print(f'  dpkt parse:       {(t_parse_end-t_parse)*1000:.1f} ms')
print(f'  Total packets:    {len(raw_packets):,}')
 
t_stage1_end = time.perf_counter()
LAT_STAGE1 = (t_stage1_end - t_stage1_start) * 1000
print(f'\\n[Stage 1] Total: {LAT_STAGE1:.1f} ms for {filesize/1e6:.1f} MB file')
print(f'  (Live: ~0 ms — stream already fixed at packet level, no file I/O)')

Loading /kaggle/input/datasets/ngtht71/vidcall/video_1_84332002263_10444-b7kd7i1u7fdq7e49rdguwtiumw.txt (1.95 MB)...
  File read:        48.2 ms
  0x20 bytes found: 6,831 (0.35% of file)
  Fix (0x20→0x00):  83.9 ms
  dpkt parse:       4.4 ms
  Total packets:    2,110
\n[Stage 1] Total: 178.2 ms for 1.9 MB file
  (Live: ~0 ms — stream already fixed at packet level, no file I/O)


In [18]:
# ==== PCAP REPLAY (multi-stream + SSRC filter) ====
import dpkt, socket, time, struct
from collections import defaultdict

def parse_rtp(data: bytes):
    if len(data) < 12: return None
    if (data[0] >> 6) != 2: return None
    cc = data[0] & 0x0F
    has_ext = (data[0] >> 4) & 1
    marker = (data[1] >> 7) & 1
    pt = data[1] & 0x7F
    seq = struct.unpack_from('>H', data, 2)[0]
    ts  = struct.unpack_from('>I', data, 4)[0]
    ssrc= struct.unpack_from('>I', data, 8)[0]
    off = 12 + cc*4
    if has_ext:
        if off + 4 > len(data): return None
        ext_len = struct.unpack_from('>H', data, off+2)[0]
        off += 4 + ext_len*4
    if off > len(data): return None
    return seq, ts, ssrc, marker, pt, data[off:]

# def replay_pcap(
#     pcap_path,
#     dst_ip="127.0.0.1", dst_port=5004,
#     ssrc_allow=None, pt_allow=None, udp_port_allow=None,
#     speed=1.0, loop=False
# ):
#     sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)

#     while True:
#         with open(pcap_path, "rb") as f:
#             pcap = dpkt.pcap.Reader(f)
#             prev_ts = None

#             for ts, buf in pcap:
#                 try:
#                     eth = dpkt.ethernet.Ethernet(buf)
#                     ip  = eth.data
#                     if not isinstance(ip, dpkt.ip.IP): continue
#                     udp = ip.data
#                     if not isinstance(udp, dpkt.udp.UDP): continue

#                     if udp_port_allow and (udp.sport not in udp_port_allow and udp.dport not in udp_port_allow):
#                         continue

#                     rtp = parse_rtp(udp.data)
#                     if rtp is None: continue
#                     seq, rtp_ts, ssrc, marker, pt, payload = rtp

#                     if ssrc_allow and ssrc not in ssrc_allow:
#                         continue
#                     if pt_allow and pt not in pt_allow:
#                         continue

#                     # timing
#                     if prev_ts is not None:
#                         delay = (ts - prev_ts) / speed
#                         if delay > 0:
#                             time.sleep(delay)
#                     prev_ts = ts

#                     sock.sendto(udp.data, (dst_ip, dst_port))
#                 except Exception:
#                     continue

#         if not loop:
#             break

class RTPReplay:

    def __init__(self, pcap_file, dst_ip, dst_port):

        self.sock = socket.socket(
            socket.AF_INET,
            socket.SOCK_DGRAM
        )

        self.dst = (dst_ip, dst_port)

        self.packets = load_packets(pcap_file)

    def replay(self):

        prev_ts = None

        for pkt_time, raw in self.packets:

            if prev_ts is not None:

                delta = pkt_time - prev_ts

                if delta > 0:
                    time.sleep(delta)

            self.sock.sendto(raw, self.dst)

            prev_ts = pkt_time

# ==== Latency measurement ====
def measure_latency(process_fn, frame):
    t0 = time.perf_counter()
    out = process_fn(frame)
    t1 = time.perf_counter()
    return out, (t1 - t0) * 1000

In [19]:
# def main():
#     parser = argparse.ArgumentParser(
#         description='IMS Video Call Real-time Filter Pipeline'
#     )
#     parser.add_argument('--listen-ip',   default='0.0.0.0')
#     parser.add_argument('--listen-port', type=int, default=5004)
#     parser.add_argument('--dest-ip',     default='127.0.0.1')
#     parser.add_argument('--dest-port',   type=int, default=5006)
#     parser.add_argument('--ssrc',        default=None,
#                         help='Target SSRC (hex), e.g. 0x5cccb090')
#     parser.add_argument('--face-model',  default='face_landmarker.task')
#     parser.add_argument('--seg-model',   default='selfie_segmenter.tflite')
#     parser.add_argument('--effect',      default='bg_blur',
#                         choices=['face_ar', 'bg_remove', 'bg_replace', 'bg_blur'])
#     parser.add_argument('--bg-image',    default='',
#                         help='Background image path (for bg_replace)')
#     parser.add_argument('--fps',         type=float, default=15.0)
#     parser.add_argument('--bitrate',     type=int,   default=500000)
#     parser.add_argument('--jitter-ms',   type=int,   default=80)
#     # parser.add_argument('--download-models', action='store_true')

#     parser.add_argument('--pcap', default=None,
#                     help='Replay RTP from PCAP file')

#     parser.add_argument('--pcap-ssrc', default=None,
#                     help='Filter SSRC (hex)')
#     args = parser.parse_args()
 
#     if args.download_models:
#         download_models()
#         return
 
#     config = PipelineConfig(
#         listen_ip        = args.listen_ip,
#         listen_port      = args.listen_port,
#         dest_ip          = args.dest_ip,
#         dest_port        = args.dest_port,
#         ssrc_in          = int(args.ssrc, 16) if args.ssrc else None,
#         face_model_path  = args.face_model,
#         seg_model_path   = args.seg_model,
#         effect           = EffectType(args.effect),
#         bg_image_path    = args.bg_image,
#         fps              = args.fps,
#         bitrate          = args.bitrate,
#         jitter_buffer_ms = args.jitter_ms,
#     )
 
#     pipeline = VideoPipeline(config)

#     pcap_replayer = None

#     if args.pcap:
#         ssrc = int(args.pcap_ssrc, 16) if args.pcap_ssrc else None
    
#         pcap_replayer = PcapReplayer(
#             args.pcap,
#             config.listen_ip,
#             config.listen_port,
#             ssrc_filter=ssrc
#         )

    
#     pipeline.start()
#     if pcap_replayer:
#         pcap_replayer.start()
     
#     print("\nPress Ctrl+C to stop...\n")
#     try:
#         while True:
#             time.sleep(1)
#     except KeyboardInterrupt:
#         print("\nStopping...")
 
#     pipeline.stop()

In [20]:
# ==== NOTEBOOK RUNNER ====

import time

# ---- CONFIG ----
pcap_path = "/kaggle/input/datasets/ngtht71/vidcall/video_1_84332002263_10444-b7kd7i1u7fdq7e49rdguwtiumw.txt"
ssrc_hex  = None   # hoặc None
effect    = "bg_blur"      # face_ar | bg_remove | bg_replace | bg_blur

# ---- BUILD CONFIG ----
config = PipelineConfig(
    listen_ip="127.0.0.1",
    listen_port=5004,
    dest_ip="127.0.0.1",   # có thể đổi nếu muốn gửi sang VLC
    dest_port=5006,
    ssrc_in=int(ssrc_hex, 16) if ssrc_hex else None,
    effect=EffectType(effect),
    face_model_path="face_landmarker.task",
    seg_model_path="selfie_segmenter.tflite",
    jitter_buffer_ms=80,
    fps=30
)

# ---- INIT PIPELINE ----
pipeline = VideoPipeline(config)

# ---- INIT PCAP REPLAYER ----
pcap_replayer = PcapReplayer(
    pcap_path,
    config.listen_ip,
    config.listen_port,
    ssrc_filter=int(ssrc_hex, 16) if ssrc_hex else None,
    speed=1.0
)

# ---- START ----
pipeline.start()
pcap_replayer.start()

print("Running pipeline from PCAP...")

# ---- RUN FOR N SECONDS ----
RUN_SECONDS = 30   # chỉnh tùy ý

try:
    time.sleep(RUN_SECONDS)
except KeyboardInterrupt:
    print("Interrupted")

# ---- STOP ----
print("Stopping...")
pcap_replayer.stop()
pipeline.stop()

[Inference] ImageSegmenter loaded: selfie_segmenter.tflite
[Packetizer] Sending to 127.0.0.1:5006, SSRC=0x00000039


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1778732692.197301     155 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


NameError: name 'LatencyTracker' is not defined

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

lat = pipeline._latency.times

plt.figure()
plt.plot(lat)
plt.title("Latency per frame (ms)")
plt.xlabel("Frame")
plt.ylabel("Latency (ms)")
plt.show()

print("Avg:", np.mean(lat))
print("P90:", np.percentile(lat, 90))
print("Max:", np.max(lat))